# 01 · Data exploration

BJTUSensor Φ-OTDR dataset (Cao et al., 2023): 1-second windows of **10,000 light pulses × 12 sensing points**, six classes. Numbers come from `ml/scripts/verify_dataset.py` and `ml/scripts/explore_data.py`; re-run those to regenerate the files read here.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"


## Class counts per split

15,419 files are listed; one test background file is empty in the source zip, leaving **15,418** usable samples.

In [2]:
pd.read_csv(RES / "dataset_summary.csv")

,split,background,dig,knock,shake,walk,water,total
0,test,588,502,506,546,490,451,3083
1,train,2357,2010,2024,2182,1960,1802,12335
2,total,2945,2512,2530,2728,2450,2253,15418


## Who recorded what

Recorders are tied to classes (e.g. shake comes almost only from `yu`), so a leave-one-recorder-out test is impossible. Phase 4 holds out whole recording sessions and days instead.

In [3]:
pd.read_csv(RES / "recorders.csv")

,recorder,background,dig,knock,shake,walk,water,total
0,cxm,2100,537,1093,0,1175,1067,5972
1,dy,0,546,174,0,0,0,720
2,jzy,0,325,0,0,0,301,626
3,ssm,0,0,127,0,0,0,127
4,sys,0,135,118,0,0,457,710
5,tml,0,105,0,0,0,0,105
6,unknown,846,712,730,778,600,428,4094
7,yu,0,40,0,1764,444,0,2248
8,yyn,0,112,288,186,0,0,586
9,yzs,0,0,0,0,231,0,231


In [4]:
bad = pd.read_csv(RES / "bad_files.csv")
bad

,path,problem
0,E:\NGN Project\ml\data\raw\test\01_background\...,not a readable .mat file: Mat file appears to ...


## One sample per class

![waterfalls](../../results/explore_waterfalls.png)

![traces](../../results/explore_traces.png)

## Where along the fiber, and at which frequencies

Event energy concentrates on points 4–9; background is flat. Background is ~100× weaker and carries a few narrow interference lines; shake and water hold the most high-frequency power.

![point std](../../results/explore_point_std.png)

![spectra](../../results/explore_spectra.png)

In [5]:
stats = pd.read_csv(RES / "explore_class_stats.csv", index_col=0)
stats[["spectral_centroid", "low_band_power_share"]]

,spectral_centroid,low_band_power_share
background,0.1023,0.7324
dig,0.1852,0.5886
knock,0.0950,0.7866
water,0.2319,0.4731
shake,0.2804,0.4085
walk,0.0985,0.7491


## The cached model input

Each window is z-scored and mean-pooled ×10 in time → (12, 1000), stored as float16 in `ml/data/processed/` (git-ignored; rebuild with `ml/scripts/build_cache.py`).

In [6]:
proc = ROOT / "ml/data/processed"
if (proc / "train_X.npy").exists():
    X = np.load(proc / "train_X.npy", mmap_mode="r")
    meta = pd.read_csv(proc / "train_meta.csv")
    print("train cache", X.shape, X.dtype, "| validation share", round(meta.is_val.mean(), 3))
else:
    print("cache not built on this machine")

train cache (12335, 12, 1000) float16 | validation share 0.1
